# AI Writer
* DWL의 Gradio 앱입니다.
* 음성 녹음을 하고 그것을 바탕으로 영어 단락을 작성해주는 일을 합니다.
* OpenAI API Key가 필요합니다.
* 연구원들은 이 앱의 사용법에 관하여 김태경 교수에게 안내받으십시오.

>2025년 8월 30일, 김태경(tk_kim@khu.ac.kr)

In [8]:
# @title
# =========================================
# Colab Interactive Paragraph Builder (Gradio)
# 요구사항 1~13 구현
# =========================================
import os, io, re, json, glob, time, uuid
from datetime import datetime
from typing import Tuple, Dict, Any, List

# 기존 코드와의 일관성 유지
import ipywidgets as widgets
from IPython.display import display, clear_output

import gradio as gr
from google.colab import userdata
from openai import OpenAI

# =============== OpenAI Client =============================
api_key = userdata.get('OPENAI_API_KEY')
client = OpenAI(api_key=api_key)

# ==================== Utility =============================
# 성별별 보이스 리스트 (사용자 제공 그대로 유지)
voices = {
    "male": ["verse", "onyx", "echo", "ballad", "ash"],
    "female": ["shimmer", "sage", "nova", "fable", "coral", "alloy"]
}

# 오디오 파일 생성 함수 (사용자 제공 함수 그대로 사용)
def create_audio_file(client, input_text, output_file="default.mp3", voice="verse", instructions="", speed=0.8):
    try:
        with client.audio.speech.with_streaming_response.create(
            model="gpt-4o-mini-tts",
            voice=voice,
            input=input_text,
            instructions=instructions,       # 스타일 프롬프트
            response_format="mp3",           # 출력 포맷
            speed=speed                      # 말하기 속도
        ) as response:
            response.stream_to_file(output_file)
        return True
    except Exception as e:
        print(f"❌ 오류 발생: {e}")
        return False

# ========================== AI (원본 유지) ============================
def response_id(func):
    def wrapper(*args, **kwargs):
        res = func(*args, **kwargs)      # 원래 함수 실행하여 결과(res) 받음
        print(res.output_text)
        return res.id
    return wrapper

@response_id
def get_response_linked(message, previous_id=None):
    """
    이전 응답 ID(previous_id)를 기반으로 이어지는 대화를 생성하는 함수.
    message: 새로 보낼 메시지
    previous_id: 이전 응답의 ID (없으면 None)
    """
    res = client.responses.create(
        model='gpt-5-nano',            # 사용할 모델 지정
        previous_response_id=previous_id, # 이어지는 대화일 경우 이전 응답 ID 전달
        input=message                     # 사용자 입력 메시지
    )
    return res

# 파이프 구조 (원본 유지)
class ChatPipe:
    def __init__(self, sender):
        self.sender = sender
        self.prev_id = None
    def __rshift__(self, message):
        self.prev_id = self.sender(message, previous_id=self.prev_id)
        return self
    def reset(self):
        self.prev_id = None
        return self
    @property
    def last_id(self):
        return self.prev_id

# ====================== New Helpers for App ======================
BASE_DIR = "/content/paragraphs"
TXT_DIR = os.path.join(BASE_DIR, "txt")
AUDIO_DIR = os.path.join(BASE_DIR, "audio")
META_PATH = os.path.join(BASE_DIR, "meta.json")

os.makedirs(TXT_DIR, exist_ok=True)
os.makedirs(AUDIO_DIR, exist_ok=True)

def _scan_indices(ext: str) -> List[int]:
    """이미 저장된 일련번호 스캔"""
    pattern = os.path.join(TXT_DIR if ext == "txt" else AUDIO_DIR, f"*.{ext}")
    files = glob.glob(pattern)
    idxs = []
    for f in files:
        m = re.search(r"(\d{3})\."+ext+"$", os.path.basename(f))
        if m:
            idxs.append(int(m.group(1)))
    return sorted(idxs)

def next_serial_index() -> int:
    """다음 저장 인덱스(정수)"""
    txt_idx = _scan_indices("txt")
    mp3_idx = _scan_indices("mp3")
    cur = max(txt_idx[-1] if txt_idx else -1, mp3_idx[-1] if mp3_idx else -1)
    return cur + 1

def padded(idx: int) -> str:
    return f"{idx:03d}"

def load_meta() -> Dict[str, Any]:
    if os.path.exists(META_PATH):
        try:
            with open(META_PATH, "r", encoding="utf-8") as f:
                return json.load(f)
        except Exception:
            pass
    return {"entries": []}

def save_meta(meta: Dict[str, Any]):
    with open(META_PATH, "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)

def last_n_entries(n=5) -> List[Dict[str, Any]]:
    meta = load_meta()
    return meta.get("entries", [])[-n:]

# ============== Audio Transcription ==============
def transcribe_audio(audio_path: str) -> str:
    """
    Colab/Gradio로 녹음된 음성 파일을 텍스트로 변환.
    우선 최신 모델(gpt-4o-transcribe), 실패 시 whisper-1로 폴백.
    """
    try:
        with open(audio_path, "rb") as f:
            tr = client.audio.transcriptions.create(
                model="gpt-4o-transcribe",
                file=f
            )
        text = getattr(tr, "text", None) or getattr(tr, "output_text", None)
        if not text:
            # 일부 SDK 응답 구조 호환
            text = tr.__dict__.get("text") or tr.__dict__.get("output_text") or ""
        return text.strip()
    except Exception:
        try:
            with open(audio_path, "rb") as f:
                tr = client.audio.transcriptions.create(
                    model="whisper-1",
                    file=f
                )
            text = getattr(tr, "text", None) or getattr(tr, "output_text", None)
            if not text:
                text = tr.__dict__.get("text") or tr.__dict__.get("output_text") or ""
            return text.strip()
        except Exception as e2:
            return f"[Transcription Error] {e2}"

# ============== Build Prompt & Call LLM ==============
def build_structured_prompt(user_core_idea: str) -> str:
    """
    요구사항 2의 형식에 맞추어 LLM에게 결과 생성을 요청하는 프롬프트를 구성.
    JSON 포맷 강제 → UI 파싱 안정성 확보
    """
    return f"""
You are an academic assistant. Read the user's recorded idea and produce three fields in strict JSON:
{{
  "understood": "AI가 이해한 요지를 한국어로 간결히 요약",
  "sentence": "학술적이고 간결한 문장 1개 (영어, SSCI급 academic tone)",
  "rationale": "해당 문장이 도출된 근거/이유 (한국어, 핵심만)"
}}

Constraints:
- "sentence" must be exactly 1 sentence in English.
- No quoted strings except JSON syntax.
- Keep it short and precise.

User Core Idea:
\"\"\"{user_core_idea}\"\"\"
"""

def call_llm_for_triplet(core_text: str) -> Dict[str, str]:
    """
    Responses API 사용 (사용자 기존 패턴 유지: client.responses)
    JSON만 반환하도록 강제. 실패 시 재시도 간단 처리.
    """
    prompt = build_structured_prompt(core_text)

    for _ in range(2):
        res = client.responses.create(
            model="gpt-5-nano",
            input=[
                {"role": "system", "content": "Return only valid JSON for downstream parsing."},
                {"role": "user", "content": prompt}
            ]
        )
        content = getattr(res, "output_text", None) or ""
        content = content.strip()
        # JSON 추출
        try:
            # 가끔 코드블록 포함 시 제거
            if content.startswith("```"):
                content = re.sub(r"^```(?:json)?\s*|\s*```$", "", content, flags=re.DOTALL)
            data = json.loads(content)
            return {
                "understood": data.get("understood", "").strip(),
                "sentence": data.get("sentence", "").strip(),
                "rationale": data.get("rationale", "").strip(),
            }
        except Exception:
            # 단순 보정
            m = re.search(r"\{.*\}", content, flags=re.DOTALL)
            if m:
                try:
                    data = json.loads(m.group(0))
                    return {
                        "understood": data.get("understood", "").strip(),
                        "sentence": data.get("sentence", "").strip(),
                        "rationale": data.get("rationale", "").strip(),
                    }
                except Exception:
                    pass
    # 실패 대비 기본값
    return {
        "understood": "[파싱 실패] 핵심 아이디어 요약을 확인해 주세요.",
        "sentence": "[Parsing failed] Please revise core idea and retry.",
        "rationale": "JSON 파싱 실패. 입력 또는 네트워크 상태를 점검하세요."
    }

# ============== Session Memory (현재 단락) ==============
class ParagraphSession:
    """
    한 단락 작성 중의 임시 메모리.
    여러 번의 녹음→생성 결과를 units에 순차적으로 누적.
    단락 저장 시 전체를 합쳐 저장 후 초기화.
    """
    def __init__(self):
        self.core_idea = ""
        self.understood = ""
        self.sentence = ""
        self.rationale = ""
        self.voice_gender = "male"
        self.voice_name = voices["male"][0]
        self.voice_instructions = ""
        self.voice_speed = 0.9
        self.last_audio_path = ""
        self.last_txt_path = ""
        self.units = []  # ← [(idx, core, understood, sentence, rationale, audio_path)]

    def add_unit(self, core, understood, sentence, rationale, audio_path=None):
        self.units.append({
            "idx": len(self.units) + 1,
            "core": (core or "").strip(),
            "understood": (understood or "").strip(),
            "sentence": (sentence or "").strip(),
            "rationale": (rationale or "").strip(),
            "audio": audio_path or ""
        })

    def paragraph_text(self) -> str:
        # 누적된 문장들을 공백 하나로 이어 단락화
        return " ".join([u["sentence"] for u in self.units if u["sentence"]]).strip()

    def units_markdown(self) -> str:
        if not self.units:
            return "진행 중 항목 없음."
        lines = ["ID | 요지(요약) | 문장(영문 1문장)", "-"*80]
        for u in self.units:
            core_short = (u["core"][:28] + "…") if len(u["core"]) > 30 else u["core"]
            sent_short = (u["sentence"][:46] + "…") if len(u["sentence"]) > 48 else u["sentence"]
            lines.append(f"{u['idx']} | {core_short} | {sent_short}")
        return "\n".join(lines)

    def clear(self):
        self.__init__()


SESSION = ParagraphSession()

# ============== Saving & History Handling ==============
def save_paragraph_and_audio() -> Tuple[str, str, Dict[str, Any]]:
    """
    a) 단락 텍스트 저장 (일련번호 000.txt → 001.txt …)
    b) 음성 저장 (일련번호 000.mp3 → 001.mp3 …)
    c) 메타 업데이트
    """
    idx = next_serial_index()
    idx_str = padded(idx)

    # 파일 경로
    txt_path = os.path.join(TXT_DIR, f"{idx_str}.txt")
    mp3_path = os.path.join(AUDIO_DIR, f"{idx_str}.mp3")

    # 누적 단락 텍스트(문장들 연결)
    full_paragraph = SESSION.paragraph_text()

    # 텍스트 저장(모든 unit 디테일 포함)
    detail_lines = []
    for u in SESSION.units:
        detail_lines.append(
            f"[녹음 {u['idx']}]"
            f"\n[핵심 아이디어]\n{u['core']}"
            f"\n[AI가 이해한 바]\n{u['understood']}"
            f"\n[학술 문장]\n{u['sentence']}"
            f"\n[이유]\n{u['rationale']}\n"
        )

    content = (
        f"[전체 단락]\n{full_paragraph}\n\n"
        + "\n".join(detail_lines).strip()
    )

    with open(txt_path, "w", encoding="utf-8") as f:
        f.write(content)

    # 음성 생성
    tts_ok = create_audio_file(
        client=client,
        input_text=full_paragraph if full_paragraph else SESSION.sentence,
        output_file=mp3_path,
        voice=SESSION.voice_name,
        instructions=SESSION.voice_instructions,
        speed=float(SESSION.voice_speed)
    )

    # 메타 업데이트
    meta = load_meta()
    entry = {
        "id": idx_str,
        "time": datetime.now().isoformat(timespec="seconds"),
        "txt": txt_path,
        "mp3": mp3_path if tts_ok else "",
        "voice": SESSION.voice_name,
        "speed": SESSION.voice_speed
    }
    meta["entries"].append(entry)
    save_meta(meta)

    SESSION.last_txt_path = txt_path
    SESSION.last_audio_path = mp3_path if tts_ok else ""

    return txt_path, (mp3_path if tts_ok else ""), entry

def get_last5_display() -> List[Dict[str, Any]]:
    """
    최근 5개 단락을 표시하기 위한 정보
    """
    return last_n_entries(5)

# ============== Gradio UI Logic ==============
STATUS_PREFIX = "진행 상태"

def run_pipeline(audio_file, voice_gender, voice_name, voice_instructions, voice_speed):
    """
    1) 음성 받아서
    2) 전사 → 핵심 아이디어 텍스트
    3) LLM 호출 → understood/sentence/rationale
    4) 편집 가능 텍스트로 반환
    """
    # 진행 상태 메시지(스크롤 과다 방지: 상태 텍스트만 갱신)
    status_msgs = []

    def step(msg):
        status_msgs.append(f"• {msg}")
        # 상태는 마지막 8줄만 유지하여 과도한 스크롤 방지
        keep = status_msgs[-8:]
        return "\n".join([f"{STATUS_PREFIX}:"] + keep)

    status = step("오디오 수신 완료. 전사 시작…")

    if audio_file is None:
        return status, "", "", "", gr.update(value=""), gr.update(value=""), gr.update(value=""), gr.update(choices=voices[voice_gender], value=voice_name), get_last5_table(), get_last5_audio()

    # Gradio는 audio_file을 {"name":, "data":} 또는 파일 경로로 전달
    if isinstance(audio_file, dict) and "name" in audio_file and "data" in audio_file:
        # bytes → temp wav 저장
        temp_path = f"/content/_rec_{uuid.uuid4().hex}.wav"
        with open(temp_path, "wb") as f:
            f.write(audio_file["data"])
        audio_path = temp_path
    elif isinstance(audio_file, str):
        audio_path = audio_file
    else:
        audio_path = str(audio_file)

    # 1) Transcribe
    core = transcribe_audio(audio_path)
    SESSION.core_idea = core
    status = step("전사 완료. LLM 분석 요청…")

    # 2) LLM call
    trip = call_llm_for_triplet(core)
    SESSION.understood = trip["understood"]
    SESSION.sentence = trip["sentence"]
    SESSION.rationale = trip["rationale"]
    status = step("LLM 분석 완료. 편집 영역 갱신…")

    # 보이스 설정 반영
    SESSION.voice_gender = voice_gender
    SESSION.voice_name = voice_name
    SESSION.voice_instructions = voice_instructions or ""
    SESSION.voice_speed = voice_speed
    SESSION.add_unit(core=SESSION.core_idea,
                     understood=SESSION.understood,
                     sentence=SESSION.sentence,
                     rationale=SESSION.rationale,
                     audio_path=audio_path)

    return (
        status,
        SESSION.core_idea,
        SESSION.understood,
        SESSION.rationale,
        gr.update(value=SESSION.sentence),
        gr.update(value=SESSION.voice_instructions),
        gr.update(value=SESSION.voice_speed),
        gr.update(choices=voices[voice_gender], value=voice_name),
        get_last5_table(),
        *get_last5_audio_updates(),
        gr.update(value=SESSION.units_markdown()),         # ⬅️ units_md
        gr.update(value=SESSION.paragraph_text()),         # ⬅️ paragraph_preview
        gr.update(value=None)                              # ⬅️ audio_in: 다음 녹음 바로 가능
    )

def update_voice_list(gender):
    return gr.update(choices=voices[gender], value=voices[gender][0])

def finalize_and_save(edited_sentence):
    """
    사용자가 문장 편집 완료 후 저장.
    텍스트/오디오 둘 다 일련번호로 저장하고
    세션 메모리 비우기 + 최근 5개 리프레시
    """
    SESSION.sentence = (edited_sentence or "").strip()
    if not SESSION.sentence:
        # 비어 있으면 저장 중단
        status = f"{STATUS_PREFIX}:\n• 저장 중단: 문장이 비어 있습니다."
        return status, "", "", get_last5_table(), *get_last5_audio_updates()

    status = f"{STATUS_PREFIX}:\n• 저장 중…"
    txt_path, mp3_path, entry = save_paragraph_and_audio()
    status += f"\n• 저장 완료: {entry['id']}"

    # 세션 메모리 비우기
    SESSION.clear()
    status += "\n• 세션 초기화 완료."

    return status, txt_path, mp3_path, get_last5_table(), *get_last5_audio_updates(), \
           gr.update(value="진행 중 항목 없음."), gr.update(value=""), gr.update(value=None)

def get_last5_table():
    rows = get_last5_display()
    if not rows:
        return "최근 단락 없음."
    lines = ["ID | Time | Voice | Speed", "-"*40]
    for r in rows:
        lines.append(f"{r['id']} | {r['time']} | {r.get('voice','')} | {r.get('speed','')}")
    return "\n".join(lines)


def get_last5_audio_updates():
    """
    Gradio 오디오 5개 컴포넌트에 줄 업데이트 객체(tuple 5개)를 반환.
    각 원소는 gr.update(value=파일경로 또는 None, label=라벨) 형태.
    """
    rows = get_last5_display()[::-1]  # 최신 먼저
    updates = []
    for i in range(5):
        if i < len(rows):
            r = rows[i]
            mp3 = r.get("mp3", "")
            path = mp3 if mp3 and os.path.exists(mp3) else None
            label = f"{r['id']} — {os.path.basename(mp3) if mp3 else 'no audio'}"
            updates.append(gr.update(value=path, label=label))
        else:
            updates.append(gr.update(value=None, label=f"최근 #{i+1}"))
    return tuple(updates)  # 길이 5


# ============== Build Gradio UI ==============
with gr.Blocks(title="Academic Paragraph Builder", css="""
#status_box { white-space: pre-wrap; font-family: ui-monospace, SFMono-Regular, Menlo, Consolas, "Liberation Mono", monospace; }
""") as demo:
    gr.Markdown("### 단락 작성 도우미 (음성→아이디어→문장 1개→저장/TTS)\n- 진행 상태 메시지는 상단 박스에서 갱신됩니다.")
    with gr.Row():
        with gr.Column(scale=1):
            status_box = gr.Textbox(label="진행 상태", value=f"{STATUS_PREFIX}:\n• 대기 중", lines=8, max_lines=8, interactive=False, elem_id="status_box")
            audio_in = gr.Audio(label="1) 사용자의 음성 녹음/업로드", sources=["microphone", "upload"], type="filepath")
            with gr.Row():
                voice_gender = gr.Radio(label="9) 성별 선택", choices=["male","female"], value="male")
                voice_name = gr.Dropdown(label="보이스 선택", choices=voices["male"], value=voices["male"][0])
            voice_instructions = gr.Textbox(label="10) 음성 스타일 프롬프트 (예: 'Formal academic tone, neutral pace')", placeholder="선택 사항", lines=2)
            voice_speed = gr.Slider(label="TTS 속도", minimum=0.5, maximum=1.3, step=0.05, value=0.9)
            run_btn = gr.Button("3) 분석 실행 (전사→LLM)", variant="primary")

            core_idea = gr.Textbox(label="[입력한 사용자의 핵심 아이디어] (자동 전사 결과, 필요 시 수정)", lines=4)
            understood = gr.Textbox(label="[AI가 이해한 바] (자동 생성, 필요 시 수정)", lines=3)
            rationale = gr.Textbox(label="[작성의 이유] (자동 생성, 필요 시 수정)", lines=3)
            edited_sentence = gr.Textbox(label="4) [학술적인 문장 1개 작성] — 편집 가능 (영어 1문장)", lines=3, placeholder="여기에 AI가 생성한 문장이 들어옵니다. 자유롭게 편집 후 저장하세요.")

            finalize_btn = gr.Button("5) 단락 종결 및 파일 저장 (txt/mp3 일련번호)", variant="primary")

        with gr.Column(scale=1):
            gr.Markdown("### 12–13) 최근 5개 단락 미리보기 + 오디오 재생")
            units_md = gr.Textbox(label="진행 중 녹음→문장 히스토리", value="진행 중 항목 없음.", lines=8, interactive=False)
            paragraph_preview = gr.Textbox(label="현재 단락 미리보기 (누적 문장)", value="", lines=4, interactive=False)
            last5_table = gr.Textbox(label="최근 5개 목록", value=get_last5_table(), lines=10, interactive=False)
            # 5개의 오디오 플레이어
            audio1 = gr.Audio(label="최근 #1", interactive=False)
            audio2 = gr.Audio(label="최근 #2", interactive=False)
            audio3 = gr.Audio(label="최근 #3", interactive=False)
            audio4 = gr.Audio(label="최근 #4", interactive=False)
            audio5 = gr.Audio(label="최근 #5", interactive=False)
            saved_txt_path = gr.Textbox(label="저장된 TXT 경로", interactive=False)
            saved_mp3_path = gr.Textbox(label="저장된 MP3 경로", interactive=False)

    # 이벤트 바인딩
    voice_gender.change(fn=lambda g: update_voice_list(g), inputs=voice_gender, outputs=voice_name)

    run_btn.click(
        fn=run_pipeline,
        inputs=[audio_in, voice_gender, voice_name, voice_instructions, voice_speed],
        outputs=[
            status_box, core_idea, understood, rationale,
            edited_sentence, voice_instructions, voice_speed, voice_name,
            last5_table,  # 표
            audio1, audio2, audio3, audio4, audio5,
            units_md,              # ⬅️ 새 출력
            paragraph_preview,     # ⬅️ 새 출력
            audio_in               # ⬅️ 오디오 입력 리셋
        ]
    )

    finalize_btn.click(
        fn=finalize_and_save,
        inputs=[edited_sentence],
        outputs=[
            status_box, saved_txt_path, saved_mp3_path, last5_table,
            audio1, audio2, audio3, audio4, audio5,
            units_md, paragraph_preview, audio_in
        ]
    )



### 다음 셀의 코드를 실행하고 웹 사이트를 열 것.

In [9]:
# @title
ret = demo.launch(
    share=True,              # 공개 URL 생성
    inline=False,            # ✅ 노트북 셀에 UI 임베드 금지 (링크만)
    inbrowser=False,         # 자동 새탭 열기 원하면 True
    prevent_thread_lock=True,# 셀이 블록되지 않게
    show_error=True,
    debug=False,
    quiet=True               # 안내문 억제
)

# URL만 깔끔하게 출력
url = getattr(ret, "share_url", None) or getattr(ret, "local_url", None)
print("Gradio URL:", url)


* Running on public URL: https://c7741eb7112eceae7b.gradio.live
Gradio URL: None


### 작업을 완료하면 Colab의 폴더를 열고 작업 내용을 다운로드 할 것. 또한 이후 아래 코드를 실행하여 서버를 종결할 것.

In [10]:
# @title
demo.close()

Closing server running on port: 7860
